# **Artifical Neural Network**

In [ ]:
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score

# 1. Upload dataset
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)

Saving data.csv to data (2).csv


In [ ]:
print("Dataset:")
print(df.head())
print("Columns:", df.columns.tolist())

# 2. Remove unnecessary columns
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

# 3. Select target column
target = input("Enter target column (e.g. diagnosis): ").strip()

df = df.dropna(subset=[target])
X = df.drop(columns=[target])
y = df[target]

# Convert categorical input columns to numeric
X = pd.get_dummies(X, drop_first=True)

# Encode target labels
y = LabelEncoder().fit_transform(y)

# Convert features to NumPy arrays
X = X.to_numpy(dtype=float)
y = y.reshape(-1, 1).astype(float)

# 4. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 6. Initialize ANN parameters
np.random.seed(42)

input_neurons = X_train.shape[1]
hidden_neurons = 10
output_neurons = 1

W1 = np.random.randn(input_neurons, hidden_neurons) * 0.1
b1 = np.zeros((1, hidden_neurons))

W2 = np.random.randn(hidden_neurons, output_neurons) * 0.1
b2 = np.zeros((1, output_neurons))

learning_rate = 0.01
epochs = 1000

# Sigmoid activation function
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

# 7. Train using backpropagation
for epoch in range(epochs):

    # Forward propagation
    z1 = X_train @ W1 + b1
    a1 = np.maximum(0, z1)  # ReLU hidden layer

    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)       # Output probability

    # Backpropagation
    m = X_train.shape[0]

    dz2 = (a2 - y_train) / m
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    dz1 = (dz2 @ W2.T) * (z1 > 0)
    dW1 = X_train.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    # Update weights and biases
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if (epoch + 1) % 200 == 0:
        print("Epoch:", epoch + 1)

# 8. Test the trained ANN
z1_test = X_test @ W1 + b1
a1_test = np.maximum(0, z1_test)
z2_test = a1_test @ W2 + b2
y_prob = sigmoid(z2_test)

y_pred = (y_prob >= 0.5).astype(int)

# 9. Display results
print("\nActual values:   ", y_test[:10].flatten())
print("Predicted values:", y_pred[:10].flatten())

accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", round(accuracy * 100, 2), "%")

Dataset:
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  s